In [4]:
# ============================================
# VAR(1) Fit + Roots(Eigenvalue) Check + Save
# ============================================

import os
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR

# =============================
# Config
# =============================
DATA_PATH = "./merged_var_input.csv"
LAG = 1

OUT_PARAMS = "./var_params_var1.csv"
OUT_ROOTS = "./var_roots_var1.csv"
OUT_RESID_COV = "./var_resid_cov_var1.csv"
OUT_SUMMARY_TXT = "./var_summary_var1.txt"

# =============================
# 1) Load data
# =============================
df = pd.read_csv(DATA_PATH)

# Date 제거(있으면)
if "Date" in df.columns:
    df = df.drop(columns=["Date"])

# diff 변수만 사용
cols = [c for c in df.columns if c.startswith("dlog_") or c.startswith("d_")]
data = df[cols].dropna()

print("사용 변수:", cols)
print("표본 수:", len(data))
print("lag:", LAG)
print("-" * 50)

# =============================
# 2) VAR 적합
# =============================
model = VAR(data)
results = model.fit(LAG)

# -----------------------------
# summary 저장 (중요)
# -----------------------------
summary_str = str(results.summary())
with open(OUT_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(summary_str)

print(summary_str[:1500])
print("\n[INFO] summary saved:", OUT_SUMMARY_TXT)

# =============================
# 3) 계수 저장
# - var_params_var1.csv: (각 방정식별) lag1 계수행렬
# =============================
coef_df = pd.DataFrame(results.coefs[0], index=cols, columns=cols)
coef_df.to_csv(OUT_PARAMS, encoding="utf-8-sig")

# =============================
# 4) Roots 저장
# - statsmodels의 results.roots 는 'inverse roots' 입니다.
# - 안정 조건: abs(root) > 1 (모두)
# =============================
roots = results.roots
roots_df = pd.DataFrame({
    "root": roots.astype(str),          # 복소수 포함 가능 -> 문자열로 저장
    "abs_root": np.abs(roots)
})
roots_df.to_csv(OUT_ROOTS, index=False, encoding="utf-8-sig")

print("\nRoots (inverse roots):")
print(roots_df)
print("Max |root|:", float(roots_df["abs_root"].max()))

# =============================
# 5) Stability 판정 (수정됨)
# =============================
is_stable_roots = bool(np.all(np.abs(roots) > 1))
is_stable_api = bool(results.is_stable(verbose=False))

print("\nStable 여부 (by roots abs>1):", is_stable_roots)
print("Stable 여부 (results.is_stable):", is_stable_api)

if is_stable_roots != is_stable_api:
    print("[WARN] roots 기반 판정과 is_stable()가 다릅니다. 설정/버전 확인 필요.")

# =============================
# 6) 잔차 공분산 저장
# =============================
resid_cov = pd.DataFrame(results.sigma_u, index=cols, columns=cols)
resid_cov.to_csv(OUT_RESID_COV, encoding="utf-8-sig")

print("\n저장 완료:")
print(" -", OUT_SUMMARY_TXT)
print(" -", OUT_PARAMS)
print(" -", OUT_ROOTS)
print(" -", OUT_RESID_COV)

사용 변수: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
표본 수: 1325
lag: 1
--------------------------------------------------
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 04, Mar, 2026
Time:                     09:34:59
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -39.1280
Nobs:                     1324.00    HQIC:                  -39.2015
Log likelihood:           16617.2    FPE:                9.03375e-18
AIC:                     -39.2456    Det(Omega_mle):     8.83181e-18
--------------------------------------------------------------------
Results for equation dlog_SOLVPN
                    coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------
const                  0.000383         0.000339            1.128      